本教程介绍 UniParser 账户与计费相关接口的用法，包括账户信息、API Key 管理、用量统计和余额查询。

本文以官网 `https://uniparser.dp.tech/` 服务为例进行演示。

这些接口均通过 `X-API-Key` 认证，用户可访问自己账号的数据：
- 账户自身信息（`/users/me`）
- API Key 管理（`/api-keys`）
- 计费与用量信息（`/billing/usage`、`/billing/usage-records`）
- 余额信息（`/balance`、`/balance/transactions`）

## 导入依赖

In [5]:
import json
import os
import requests

In [6]:
import os
from getpass import getpass

os.environ["UNIPARSER_API_KEY"] = getpass("请输入 Uni-Parser API Key: ")

In [7]:
host = "https://uniparser.dp.tech"  # 官网

api_key = os.getenv("UNIPARSER_API_KEY")
if not api_key:
    raise RuntimeError("请先设置 UNIPARSER_API_KEY 环境变量")

headers = {"X-API-Key": api_key}
timeout = 30


def get(path, **params):
    """账户/计费相关接口统一走 GET + X-API-Key"""
    resp = requests.get(f"{host}{path}", headers=headers, params=params or None, timeout=timeout)
    resp.raise_for_status()
    return resp.json()


def show(title, data):
    print(f"\n{'=' * 60}\n{title}\n{'=' * 60}")
    print(json.dumps(data, indent=4, ensure_ascii=False, default=str))

## Step.01 账户自身信息
> `GET /users/me` 返回当前用户基本信息 + 资料；`PUT /users/me` 更新自己的资料字段（示例见下方注释）。

In [8]:
profile = get("/users/me")
show("账户信息 (GET /users/me)", profile)

# 更新资料示例（部分字段可选，取消注释后可用）
# updated_profile = requests.put(
#     f"{host}/users/me",
#     headers=headers,
#     json={"timezone": "Asia/Shanghai", "language": "zh-CN"},
#     timeout=timeout,
# ).json()
# show("更新后的账户信息 (PUT /users/me)", updated_profile)


账户信息 (GET /users/me)
{
    "username": "demo_yaoming",
    "email": "demo_yaoming@guest.example.com",
    "is_active": true,
    "is_superuser": false,
    "id": "4765260b-4f96-48ba-93f3-7fdab33cecf8",
    "created_at": "2026-07-17T03:17:44",
    "updated_at": "2026-07-17T08:58:57",
    "status": "active",
    "rate_limit": "50/minute",
    "balance": "483.05",
    "permissions": {
        "chart": 2,
        "table": 2,
        "figure": 0,
        "textual": 2,
        "equation": 2,
        "molecule": 1,
        "pdf_pages": 6000,
        "expression": 1,
        "file_size_mb": 300
    },
    "profile": null
}


## Step.02 API Key 管理（仅本人的 key）
> `GET /api-keys` 列出全部 key（不含明文）；创建、解密和撤销都会改变或暴露敏感信息，因此示例默认不执行。

In [9]:
api_keys = get("/api-keys")
show("API Key 列表 (GET /api-keys)", api_keys)

# 如确需查看现有 Key 的明文，请在确认安全影响后手动调用：
# key_id = api_keys[0]["id"]
# decrypted = get(f"/api-keys/{key_id}/decrypt")

# 创建新 key 示例（取消注释后可用，raw_key 仅在响应中返回一次）
# new_key = requests.post(
#     f"{host}/api-keys",
#     headers=headers,
#     json={"key_identifier": "my_script_key", "permissions": []},
#     timeout=timeout,
# ).json()
# show("新建 Key (POST /api-keys)", new_key)

# 撤销 key 示例（取消注释后可用）
# requests.delete(f"{host}/api-keys/{key_id}", headers=headers, timeout=timeout)


API Key 列表 (GET /api-keys)
[
    {
        "id": "52b9db67-64c9-4b42-9383-dc0323d835a7",
        "name": "common",
        "key_identifier": "common",
        "permissions": [],
        "status": "active",
        "expires_at": "2036-07-17T03:17:44",
        "last_used_at": null,
        "created_at": "2026-07-17T03:17:44"
    }
]


## Step.03 计费与用量信息（仅本人数据）
> `GET /billing/usage` 按周期（`current_month` / `last_month` / 默认近 30 天）汇总请求数、费用、Token、页数；`GET /billing/usage-records` 分页返回最近 2 周的逐条用量明细。

In [10]:
period = "current_month"  # current_month / last_month / 其他值默认近 30 天
usage_summary = get("/billing/usage", period=period)
show(f"用量汇总 (GET /billing/usage?period={period})", usage_summary)


用量汇总 (GET /billing/usage?period=current_month)
{
    "total_requests": 23,
    "total_cost": "26.95",
    "total_tokens": 0,
    "total_pages": 0,
    "usage_by_service": {
        "parser": {
            "requests": 23,
            "cost": 26.949999999999992,
            "tokens": 0,
            "pages": 0
        }
    },
    "usage_by_date": {
        "2026-07-17": {
            "requests": 23,
            "cost": 26.949999999999992,
            "tokens": 0,
            "pages": 0
        }
    },
    "period_start": "2026-07-01T00:00:00",
    "period_end": "2026-07-31T23:59:59.999999"
}


In [11]:
page, size = 1, 10
usage_records = get("/billing/usage-records", page=page, size=size)
show(f"用量明细 (GET /billing/usage-records, 共 {usage_records.get('total', 0)} 条，仅近 2 周)", usage_records)


用量明细 (GET /billing/usage-records, 共 23 条，仅近 2 周)
{
    "items": [
        {
            "request_id": "8f8ff1e6-c573-4ba6-8945-54ed16d5e83d",
            "endpoint": "/trigger-file-async",
            "method": "POST",
            "service_type": "parser",
            "resource_usage": {
                "md5": "a349fad16277a338a822f777f22b630b",
                "page": 9.0,
                "token": "78f5e7c97e5a5fa3b66962d75f81149d",
                "dict_cfg": "{'chart': -1, 'equation': 1, 'expression': -1, 'figure': -1, 'molecule': 1, 'table': 1, 'textual': 3}",
                "proc_mol": 0.0,
                "proc_equa": 2.0,
                "proc_expr": 0.0,
                "proc_page": 9.0,
                "dict_model": "{'admin_debug': False, 'inplace_update': False, 'lang': 'unknown', 'ordering_method': 'gap_tree', 'padding_snip': True, 'pages': None, 'table_cls': False, 'timeout': 540}",
                "proc_chart": 9.0,
                "proc_table": 9.0,
                "pr

## Step.04 余额信息（仅本人数据）
> `GET /balance` 返回当前余额、货币、账户状态等；`GET /balance/transactions` 分页返回余额流水。自助充值 `POST /balance/top-up` 已禁用（返回 410），需联系管理员。

In [12]:
balance = get("/balance")
show("余额 (GET /balance)", balance)


余额 (GET /balance)
{
    "balance": "483.05",
    "currency": "CNY",
    "user_id": "4765260b-4f96-48ba-93f3-7fdab33cecf8",
    "username": "demo_yaoming",
    "status": "active",
    "bohrium_user_id": null,
    "permissions": {
        "chart": 2,
        "table": 2,
        "figure": 0,
        "textual": 2,
        "equation": 2,
        "molecule": 1,
        "pdf_pages": 6000,
        "expression": 1,
        "file_size_mb": 300
    }
}


In [13]:
page, size = 1, 10
txns = get("/balance/transactions", page=page, size=size)
show(f"余额流水 (GET /balance/transactions, 共 {txns.get('total', 0)} 条)", txns)


余额流水 (GET /balance/transactions, 共 24 条)
{
    "items": [
        {
            "transaction_id": "e90d44a7-9d92-46e8-a689-d045604dc237",
            "user_id": "4765260b-4f96-48ba-93f3-7fdab33cecf8",
            "admin_id": null,
            "amount": "-0.45",
            "balance_before": "483.50",
            "balance_after": "483.05",
            "transaction_type": "usage_deduction",
            "description": "API 使用扣费",
            "reference_id": "b4d4351c-706a-4133-9755-758d47aae4e5",
            "created_at": "2026-07-17T08:58:57"
        },
        {
            "transaction_id": "3b3b8408-94b3-4b6a-84be-6bc0ffe04d37",
            "user_id": "4765260b-4f96-48ba-93f3-7fdab33cecf8",
            "admin_id": null,
            "amount": "-0.45",
            "balance_before": "483.95",
            "balance_after": "483.50",
            "transaction_type": "usage_deduction",
            "description": "API 使用扣费",
            "reference_id": "f939b76e-e402-4458-bc0b-87b5b2c26e52",
